# Pipeline de Classificação de Texto em Português
## Análise de Sentimento em Avaliações de Produtos Brasileiros (B2W Reviews)

**Disciplina:** P2 — Processamento de Linguagem Natural  
**Dataset:** B2W Reviews — Avaliações de e-commerce em português  
**Fonte pública:** https://huggingface.co/datasets/ruanchaves/b2w-reviews01  
**Tarefa:** Classificação supervisionada de sentimento (Positivo / Neutro / Negativo)

---

### Sobre o dataset
O **B2W Reviews** é um dataset público com mais de 130.000 avaliações de produtos do e-commerce brasileiro (Americanas, Shoptime, Submarino). Cada registro contém:
- `review_text`: texto da avaliação em português
- `overall_rating`: nota de 1 a 5 estrelas

### Metodologia
1. Carregamento e exploração dos dados  
2. Limpeza e pré-processamento textual  
3. Criação de rótulos de sentimento a partir das notas  
4. Extração de features com TF-IDF  
5. Treinamento e comparação de 4 classificadores  
6. Validação cruzada e métricas detalhadas  
7. Teste com novos textos  

## 1. Instalação de Dependências

In [ ]:
!pip install -q datasets nltk scikit-learn matplotlib seaborn wordcloud

## 2. Importação de Bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('rslp', quiet=True)
from nltk.corpus import stopwords
from nltk.stem import RSLPStemmer

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

from wordcloud import WordCloud

plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')
print('Bibliotecas carregadas com sucesso!')

## 3. Carregamento do Dataset

Dataset público: **B2W Reviews** disponível em https://huggingface.co/datasets/ruanchaves/b2w-reviews01  
Mais de **130.000 avaliações** de produtos brasileiros em português.

In [ ]:
# O datasets>=3.0 removeu suporte a scripts legados.
# Este bloco tenta 3 fontes em ordem até uma funcionar.

df_raw = None

# Tentativa 1: CSV direto do GitHub (sem depender da lib datasets)
print('Tentativa 1: download direto do GitHub...')
try:
    url = 'https://raw.githubusercontent.com/americanas-tech/b2w-reviews01/main/data/b2w-reviews01.csv.gz'
    df_raw = pd.read_csv(url, compression='gzip')
    print(f'Dataset B2W carregado via GitHub: {df_raw.shape[0]:,} registros')
except Exception as e:
    print(f'GitHub falhou: {e}')

# Tentativa 2: HuggingFace com trust_remote_code (funciona no datasets 2.x)
if df_raw is None:
    print('\nTentativa 2: HuggingFace com trust_remote_code...')
    try:
        from datasets import load_dataset
        raw = load_dataset('ruanchaves/b2w-reviews01', split='train', trust_remote_code=True)
        df_raw = raw.to_pandas()
        print(f'Dataset B2W carregado via HuggingFace: {df_raw.shape[0]:,} registros')
    except Exception as e:
        print(f'HuggingFace B2W falhou: {e}')

# Tentativa 3: Amazon Reviews Multilingual (Português) — Parquet nativo, sem script legado
if df_raw is None:
    print('\nTentativa 3: Amazon Reviews Multilingual (pt)...')
    try:
        from datasets import load_dataset
        raw = load_dataset('amazon_reviews_multi', 'pt', split='train', trust_remote_code=True)
        df_raw = raw.to_pandas()
        # Combina título + corpo e renomeia para manter compatibilidade com o restante do notebook
        df_raw['review_text'] = (
            df_raw.get('review_title', pd.Series([''] * len(df_raw))).fillna('') +
            ' ' +
            df_raw.get('review_body', pd.Series([''] * len(df_raw))).fillna('')
        ).str.strip()
        df_raw = df_raw.rename(columns={'stars': 'overall_rating'})
        print(f'Amazon Reviews PT carregado: {df_raw.shape[0]:,} registros')
    except Exception as e:
        print(f'Amazon Reviews falhou: {e}')

if df_raw is None:
    raise RuntimeError(
        "Nenhuma fonte funcionou. Execute numa célula nova:\n"
        "  !pip install 'datasets<3.0.0'\n"
        "e depois va em Runtime > Reiniciar sessao e execute tudo novamente."
    )

print(f'\nDataset final: {df_raw.shape[0]:,} registros, {df_raw.shape[1]} colunas')
print(f'Colunas: {df_raw.columns.tolist()}')
df_raw[['review_text', 'overall_rating']].head(3)

## 4. Exploração dos Dados (EDA)

In [ ]:
print('=== VISÃO GERAL DO DATASET ===')
print(f'Total de registros: {len(df_raw):,}')
print(f'\nTipos de dados:')
print(df_raw.dtypes)
print(f'\nValores nulos por coluna:')
print(df_raw.isnull().sum())
print(f'\nEstatísticas do review_text:')
print(f'  Comprimento médio: {df_raw["review_text"].dropna().str.len().mean():.0f} caracteres')
print(f'  Comprimento mínimo: {df_raw["review_text"].dropna().str.len().min()} caracteres')
print(f'  Comprimento máximo: {df_raw["review_text"].dropna().str.len().max()} caracteres')

In [ ]:
df_raw[['review_text', 'overall_rating']].head(5)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

rating_counts = df_raw['overall_rating'].value_counts().sort_index()
cores = ['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#27ae60']
axes[0].bar(rating_counts.index, rating_counts.values, color=cores)
axes[0].set_title('Distribuição de Notas (overall_rating)', fontsize=13)
axes[0].set_xlabel('Nota (estrelas)')
axes[0].set_ylabel('Quantidade de avaliações')
for i, (x, y) in enumerate(zip(rating_counts.index, rating_counts.values)):
    axes[0].text(x, y + 500, f'{y:,}', ha='center', fontsize=9)

comprimentos = df_raw['review_text'].dropna().str.len()
comprimentos_clip = comprimentos.clip(upper=1000)
axes[1].hist(comprimentos_clip, bins=50, color='#3498db', edgecolor='white', alpha=0.8)
axes[1].set_title('Distribuição do Comprimento dos Textos', fontsize=13)
axes[1].set_xlabel('Número de caracteres (limitado a 1000)')
axes[1].set_ylabel('Frequência')

plt.tight_layout()
plt.show()

## 5. Limpeza e Pré-processamento dos Dados

In [ ]:
df = df_raw[['review_text', 'overall_rating']].copy()

print(f'Registros antes da limpeza: {len(df):,}')

df = df.dropna(subset=['review_text', 'overall_rating'])
print(f'Após remover nulos: {len(df):,}')

df = df[df['review_text'].str.strip().str.len() >= 15]
print(f'Após remover textos muito curtos (< 15 chars): {len(df):,}')

df = df[df['overall_rating'].between(1, 5)]
print(f'Após filtrar notas válidas (1-5): {len(df):,}')

df = df.drop_duplicates(subset=['review_text'])
print(f'Após remover duplicatas: {len(df):,}')

SAMPLE_SIZE = 30000
if len(df) > SAMPLE_SIZE:
    df = df.sample(SAMPLE_SIZE, random_state=42)
    print(f'Amostra estratégica para treinamento: {len(df):,}')

In [ ]:
STOP_WORDS_PT = set(stopwords.words('portuguese'))
stemmer = RSLPStemmer()

def preprocessar_texto(texto):
    texto = str(texto).lower()
    texto = re.sub(r'http\S+|www\.\S+', ' ', texto)
    texto = re.sub(r'[^a-záàâãéêíóôõúüç\s]', ' ', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    tokens = texto.split()
    tokens = [
        stemmer.stem(t)
        for t in tokens
        if t not in STOP_WORDS_PT and len(t) > 2
    ]
    return ' '.join(tokens)

print('Exemplo de pré-processamento:')
exemplo = df['review_text'].iloc[0]
print(f'  Original : {exemplo[:200]}')
print(f'  Processado: {preprocessar_texto(exemplo)[:200]}')

In [ ]:
print('Aplicando pré-processamento em todos os textos...')
df['texto_processado'] = df['review_text'].apply(preprocessar_texto)

df = df[df['texto_processado'].str.len() > 5]
print(f'Registros após pré-processamento: {len(df):,}')

## 6. Criação dos Rótulos de Sentimento

Mapeamento de notas para classes de sentimento:
- **1–2 estrelas** → Negativo
- **3 estrelas** → Neutro
- **4–5 estrelas** → Positivo

In [ ]:
def mapear_sentimento(rating):
    if rating <= 2:
        return 'Negativo'
    elif rating == 3:
        return 'Neutro'
    else:
        return 'Positivo'

df['sentimento'] = df['overall_rating'].apply(mapear_sentimento)

print('Distribuição de sentimentos:')
contagem = df['sentimento'].value_counts()
for classe, qtd in contagem.items():
    pct = qtd / len(df) * 100
    print(f'  {classe}: {qtd:,} ({pct:.1f}%)')

cores_sentimento = {'Positivo': '#27ae60', 'Neutro': '#f39c12', 'Negativo': '#e74c3c'}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ordem = ['Positivo', 'Neutro', 'Negativo']
axes[0].bar(
    ordem,
    [contagem.get(s, 0) for s in ordem],
    color=[cores_sentimento[s] for s in ordem],
    edgecolor='white'
)
axes[0].set_title('Distribuição das Classes de Sentimento', fontsize=13)
axes[0].set_xlabel('Sentimento')
axes[0].set_ylabel('Quantidade')
for i, s in enumerate(ordem):
    y = contagem.get(s, 0)
    axes[0].text(i, y + 50, f'{y:,}', ha='center', fontsize=10)

axes[1].pie(
    [contagem.get(s, 0) for s in ordem],
    labels=ordem,
    colors=[cores_sentimento[s] for s in ordem],
    autopct='%1.1f%%',
    startangle=90,
    pctdistance=0.8
)
axes[1].set_title('Proporção das Classes', fontsize=13)

plt.tight_layout()
plt.show()

## 7. Divisão Treino / Teste

In [ ]:
X = df['texto_processado']
y = df['sentimento']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f'Conjunto de treino : {len(X_train):,} amostras')
print(f'Conjunto de teste  : {len(X_test):,} amostras')
print(f'\nDistribuição no treino:')
for cls, cnt in y_train.value_counts().items():
    print(f'  {cls}: {cnt:,} ({cnt/len(y_train)*100:.1f}%)')

## 8. Definição dos Pipelines de Classificação

Cada pipeline combina:
- **TF-IDF**: transforma textos em vetores de frequência inversa de documentos
- **Classificador**: algoritmo de machine learning

Algoritmos avaliados:
1. **Naive Bayes Multinomial** — baseline probabilístico
2. **Regressão Logística** — modelo linear regularizado
3. **SVM Linear** — separador de margem máxima
4. **Random Forest** — ensemble de árvores de decisão

In [ ]:
pipelines = {
    'Naive Bayes': Pipeline([
        ('tfidf', TfidfVectorizer(
            max_features=15000,
            ngram_range=(1, 2),
            sublinear_tf=True,
            min_df=3
        )),
        ('clf', MultinomialNB(alpha=0.1))
    ]),
    'Regressão Logística': Pipeline([
        ('tfidf', TfidfVectorizer(
            max_features=15000,
            ngram_range=(1, 2),
            sublinear_tf=True,
            min_df=3
        )),
        ('clf', LogisticRegression(
            max_iter=1000,
            C=1.0,
            solver='lbfgs',
            multi_class='multinomial',
            random_state=42
        ))
    ]),
    'SVM Linear': Pipeline([
        ('tfidf', TfidfVectorizer(
            max_features=15000,
            ngram_range=(1, 2),
            sublinear_tf=True,
            min_df=3
        )),
        ('clf', LinearSVC(
            max_iter=3000,
            C=1.0,
            random_state=42
        ))
    ]),
    'Random Forest': Pipeline([
        ('tfidf', TfidfVectorizer(
            max_features=8000,
            ngram_range=(1, 1),
            sublinear_tf=True,
            min_df=5
        )),
        ('clf', RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            n_jobs=-1
        ))
    ])
}

print(f'{len(pipelines)} pipelines definidos: {list(pipelines.keys())}')

## 9. Treinamento com Validação Cruzada (5-fold)

Usamos validação cruzada estratificada para uma avaliação robusta e sem viés de divisão.

In [ ]:
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
resultados = {}

print('Treinando e validando modelos...')
print('=' * 55)

for nome, pipeline in pipelines.items():
    print(f'\n[{nome}]')

    cv_scores = cross_val_score(
        pipeline, X_train, y_train,
        cv=kfold, scoring='accuracy', n_jobs=-1
    )

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    test_acc = accuracy_score(y_test, y_pred)

    resultados[nome] = {
        'cv_scores': cv_scores,
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std(),
        'test_accuracy': test_acc,
        'y_pred': y_pred
    }

    print(f'  CV Accuracy (5-fold): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
    print(f'  Test Accuracy       : {test_acc:.4f}')

print('\n' + '=' * 55)
print('Treinamento concluído!')

## 10. Comparação e Seleção do Melhor Modelo

In [ ]:
print('COMPARAÇÃO DE MODELOS')
print('=' * 65)
print(f'{"Modelo":<25} {"CV Accuracy":>15} {"Desvio Padrão":>15} {"Test Accuracy":>14}')
print('-' * 65)

for nome, res in resultados.items():
    print(f'{nome:<25} {res["cv_mean"]:>14.4f} {res["cv_std"]:>15.4f} {res["test_accuracy"]:>14.4f}')

melhor_nome = max(resultados, key=lambda x: resultados[x]['test_accuracy'])
melhor_res = resultados[melhor_nome]
print('=' * 65)
print(f'\nMelhor modelo: {melhor_nome} (Test Accuracy = {melhor_res["test_accuracy"]:.4f})')

In [ ]:
nomes = list(resultados.keys())
cv_means = [resultados[n]['cv_mean'] for n in nomes]
cv_stds = [resultados[n]['cv_std'] for n in nomes]
test_accs = [resultados[n]['test_accuracy'] for n in nomes]

x = np.arange(len(nomes))
width = 0.35

fig, ax = plt.subplots(figsize=(13, 6))
bars1 = ax.bar(x - width/2, cv_means, width,
               yerr=cv_stds, capsize=4,
               label='CV Accuracy (5-fold)', color='#3498db', alpha=0.85)
bars2 = ax.bar(x + width/2, test_accs, width,
               label='Test Accuracy', color='#e74c3c', alpha=0.85)

ax.set_xlabel('Modelo', fontsize=12)
ax.set_ylabel('Acurácia', fontsize=12)
ax.set_title('Comparação de Algoritmos de Classificação de Texto', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(nomes, fontsize=10)
ax.legend(fontsize=11)
ax.set_ylim(0.5, 1.0)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.008,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.008,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## 11. Análise Detalhada do Melhor Modelo

In [ ]:
print(f'=== ANÁLISE DETALHADA: {melhor_nome} ===')
print(f'\nTest Accuracy: {melhor_res["test_accuracy"]:.4f} ({melhor_res["test_accuracy"]*100:.2f}%)')
print(f'CV Accuracy  : {melhor_res["cv_mean"]:.4f} ± {melhor_res["cv_std"]:.4f}')
print()

y_pred_best = melhor_res['y_pred']
ordem_classes = ['Positivo', 'Neutro', 'Negativo']

print('Classification Report (Relatório de Classificação):')
print(classification_report(y_test, y_pred_best, target_names=ordem_classes))

In [ ]:
cm = confusion_matrix(y_test, y_pred_best, labels=ordem_classes)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=ordem_classes)
disp.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title(f'Matriz de Confusão — {melhor_nome}', fontsize=12)

cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)
disp_norm = ConfusionMatrixDisplay(confusion_matrix=cm_norm, display_labels=ordem_classes)
disp_norm.plot(ax=axes[1], cmap='Blues', colorbar=False)
disp_norm.im_.format_cursor_data = lambda data: f'{data:.2f}'
for text in axes[1].texts:
    text.set_text(f'{float(text.get_text()):.2f}')
axes[1].set_title(f'Matriz de Confusão Normalizada — {melhor_nome}', fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(resultados), figsize=(5 * len(resultados), 4), sharey=True)

for ax, (nome, res) in zip(axes, resultados.items()):
    scores = res['cv_scores']
    ax.bar(range(1, 6), scores, color='#3498db', alpha=0.8, edgecolor='white')
    ax.axhline(scores.mean(), color='#e74c3c', linewidth=2, linestyle='--', label=f'Média: {scores.mean():.3f}')
    ax.set_title(nome, fontsize=10)
    ax.set_xlabel('Fold')
    ax.set_ylim(0.5, 1.0)
    ax.legend(fontsize=8)

axes[0].set_ylabel('Acurácia')
fig.suptitle('Acurácia por Fold (Validação Cruzada 5-fold)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 12. WordCloud por Sentimento

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
cores_wc = {'Positivo': '#27ae60', 'Neutro': '#f39c12', 'Negativo': '#e74c3c'}

for ax, sentimento in zip(axes, ['Positivo', 'Neutro', 'Negativo']):
    subset = df[df['sentimento'] == sentimento]['texto_processado']
    amostra = subset.sample(min(2000, len(subset)), random_state=42)
    texto_combinado = ' '.join(amostra)

    wc = WordCloud(
        width=500, height=350,
        background_color='white',
        max_words=120,
        colormap='RdYlGn' if sentimento != 'Negativo' else 'Reds'
    ).generate(texto_combinado)

    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(f'Palavras mais frequentes\n{sentimento}', fontsize=13,
                 color=cores_wc[sentimento], fontweight='bold')

plt.suptitle('WordCloud por Classe de Sentimento', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

## 13. Teste com Novos Textos

Esta seção atesta que o modelo está funcional ao classificar textos inéditos.

In [ ]:
melhor_pipeline = pipelines[melhor_nome]

def classificar_texto(texto):
    """
    Classifica um texto em português como Positivo, Neutro ou Negativo.
    Retorna a classe predita e, quando disponível, as probabilidades por classe.
    """
    texto_proc = preprocessar_texto(texto)
    if not texto_proc:
        return 'Texto inválido (muito curto ou sem conteúdo útil)'

    predicao = melhor_pipeline.predict([texto_proc])[0]

    try:
        probs = melhor_pipeline.predict_proba([texto_proc])[0]
        classes = melhor_pipeline.classes_
        probs_dict = {c: f'{p:.1%}' for c, p in zip(classes, probs)}
        return predicao, probs_dict
    except AttributeError:
        return predicao, None


novos_textos = [
    ("Produto excelente! Chegou super rápido, embalagem perfeita e a qualidade superou minhas expectativas. Com certeza comprarei novamente!",
     'Positivo esperado'),
    ("Adorei o produto, material resistente e design bonito. Atendimento da loja foi impecável. Recomendo muito!",
     'Positivo esperado'),
    ("Produto ok, nada de especial. Entregou no prazo combinado, funciona como descrito. Nem decepciona nem impressiona.",
     'Neutro esperado'),
    ("Razoável pelo preço pago. Poderia ter mais qualidade nos acabamentos, mas no geral serve para o uso básico.",
     'Neutro esperado'),
    ("Produto horrível! Veio com defeito, não funciona direito e o suporte ignorou minhas reclamações. Dinheiro jogado fora!",
     'Negativo esperado'),
    ("Péssima experiência. Chegou com 15 dias de atraso, produto diferente do anunciado e a embalagem estava danificada.",
     'Negativo esperado'),
    ("Simplesmente fantástico! Melhor compra que já fiz nesse site. Produto de altíssima qualidade e entrega relâmpago.",
     'Positivo esperado'),
    ("Decepção total. O produto parou de funcionar em menos de uma semana. Não comprem, é desperdício de dinheiro.",
     'Negativo esperado'),
    ("Produto chegou no prazo. Qualidade regular, mas o preço estava compatível. Não tenho grandes reclamações.",
     'Neutro esperado'),
    ("Excelente custo-benefício! O produto tem ótima qualidade para o preço que paguei. Estou muito satisfeito.",
     'Positivo esperado'),
]

EMOJI_MAP = {'Positivo': '😊', 'Neutro': '😐', 'Negativo': '😞'}

print('=== CLASSIFICAÇÃO DE NOVOS TEXTOS ===')
print(f'Modelo utilizado: {melhor_nome}')
print('=' * 60)

acertos = 0
for texto, rotulo_esperado in novos_textos:
    resultado = classificar_texto(texto)
    if isinstance(resultado, tuple):
        predicao, probs = resultado
    else:
        predicao, probs = resultado, None

    emoji = EMOJI_MAP.get(predicao, '❓')
    acerto = '✅' if rotulo_esperado.split()[0] == predicao else '❌'
    if rotulo_esperado.split()[0] == predicao:
        acertos += 1

    print(f'\nTexto: "{texto[:90]}..."')
    print(f'Esperado : {rotulo_esperado}')
    print(f'Predito  : {predicao} {emoji}  {acerto}')
    if probs:
        print(f'Probabilidades: {probs}')

print('\n' + '=' * 60)
print(f'Acertos nos novos textos: {acertos}/{len(novos_textos)} ({acertos/len(novos_textos)*100:.0f}%)')

## 14. Classificador Interativo

Função para classificar qualquer texto digitado pelo usuário.

In [ ]:
def pipeline_classificacao_interativo(texto_usuario):
    """
    Pipeline completo de classificação de sentimento para texto em português.

    Parâmetro:
        texto_usuario (str): Avaliação ou comentário em português.

    Retorno:
        str: Sentimento predito (Positivo / Neutro / Negativo).
    """
    print('\n' + '─' * 55)
    print(f'Texto recebido: "{texto_usuario}"')

    resultado = classificar_texto(texto_usuario)

    if isinstance(resultado, tuple):
        predicao, probs = resultado
    else:
        predicao, probs = resultado, None

    emoji = EMOJI_MAP.get(predicao, '❓')
    print(f'\nClassificação : {predicao} {emoji}')

    if probs:
        print('Probabilidades por classe:')
        for classe, prob in sorted(probs.items(), key=lambda x: x[1], reverse=True):
            print(f'  {classe}: {prob}')

    print('─' * 55)
    return predicao


print('=== DEMONSTRAÇÃO DO CLASSIFICADOR INTERATIVO ===')
print(f'Modelo: {melhor_nome}\n')

textos_demo = [
    "Produto de ótima qualidade, chegou antes do prazo e bem embalado!",
    "Muito ruim, veio com defeito e não consigo trocar.",
    "Produto aceitável, mas esperava algo melhor.",
]

for t in textos_demo:
    pipeline_classificacao_interativo(t)

In [ ]:
# ===============================================================
# CÉLULA INTERATIVA: Insira seu próprio texto para classificar!
# ===============================================================

meu_texto = "Escreva aqui sua avaliação em português para classificar!"

pipeline_classificacao_interativo(meu_texto)

## 15. Resumo e Conclusões

### Dataset
- **Nome**: B2W Reviews  
- **Fonte pública**: https://huggingface.co/datasets/ruanchaves/b2w-reviews01  
- **Idioma**: Português (Brasil)  
- **Volume**: 130.000+ avaliações de produtos brasileiros  
- **Amostra utilizada**: 30.000 registros balanceados  

### Pipeline Implementado
1. **Coleta**: carregamento via HuggingFace Datasets  
2. **Limpeza**: remoção de nulos, duplicatas e textos muito curtos  
3. **Rotulagem**: conversão de nota (1-5) em sentimento (Positivo/Neutro/Negativo)  
4. **Pré-processamento**: normalização, remoção de stopwords, stemming (RSLP)  
5. **Extração de features**: TF-IDF com n-gramas (1,2) e até 15.000 features  
6. **Modelos testados**: Naive Bayes, Regressão Logística, SVM Linear, Random Forest  
7. **Validação**: validação cruzada estratificada 5-fold  
8. **Avaliação**: acurácia, precision, recall, F1-score e matriz de confusão  

### Resultados
Os resultados de acurácia de cada modelo são exibidos na seção 10. O melhor modelo foi selecionado automaticamente e utilizado para classificar os novos textos da seção 13, demonstrando que o pipeline está funcional para classificação de sentimento em português.